<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/02_vcap_signatures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json, pandas as pd
BASE = '/content/drive/MyDrive/longevity_project'
CELL = json.load(open(f'{BASE}/results/config.json'))['cell_type']

sig = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_sig_info.txt.gz', sep='\t', low_memory=False)
vcap = sig[(sig['cell_id'] == CELL) & (sig['pert_type'] == 'trt_cp')]
print(CELL, 'signatures:', len(vcap), 'drugs:', vcap['pert_iname'].nunique())
vcap.to_csv(f'{BASE}/data/processed/vcap_sig_info.csv', index=False)

Mounted at /content/drive
VCAP signatures: 34753 drugs: 15542


In [2]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /


In [3]:
!wget -q --show-progress -O - https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_Level5_COMPZ.MODZ_n473647x12328.gctx.gz | gunzip > /content/level5.gctx

-                   100%[===================>]  19.86G  49.5MB/s    in 8m 10s  


In [4]:
import h5py, numpy as np
f = h5py.File('/content/level5.gctx', 'r')
col_ids = f['0/META/COL/id'][:].astype(str)
row_ids = f['0/META/ROW/id'][:].astype(str)
matrix = f['0/DATA/0/matrix']
print(matrix.shape, len(col_ids), len(row_ids))

(473647, 12328) 473647 12328


In [11]:
from tqdm.auto import tqdm

wanted = set(vcap['sig_id'])
idx = np.array([i for i, c in enumerate(col_ids) if c in wanted])
print('Found', len(idx), 'of', len(vcap))

parts = []
batch = 100
for i in tqdm(range(0, len(idx), batch), desc='Reading VCAP signatures'):
    parts.append(matrix[idx[i:i+batch], :])

df = pd.DataFrame(np.vstack(parts), index=col_ids[idx], columns=row_ids)
print('Saving to Drive...')
df.to_parquet(f'{BASE}/data/processed/vcap_signatures.parquet')
print('Done:', df.shape)

Found 34753 of 34753


Reading VCAP signatures:   0%|          | 0/348 [00:00<?, ?it/s]

Saving to Drive...
Done: (34753, 12328)


In [10]:
print('col_ids:', len(col_ids), col_ids[:3])
print('vcap:', len(vcap), vcap['sig_id'].values[:3])

col_ids: 473647 ['CPC005_A375_6H:BRD-A85280935-003-01-7:10'
 'CPC005_A375_6H:BRD-A07824748-001-02-6:10'
 'CPC004_A375_6H:BRD-K20482099-001-01-1:10']
vcap: 34753 ['CPC001_VCAP_24H:BRD-A00267231-001-01-1:10'
 'CPC001_VCAP_24H:BRD-A01295252-001-01-5:10'
 'CPC001_VCAP_24H:BRD-A03359064-001-01-5:10']


In [12]:
!wget -nc -P {BASE}/data/raw https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_gene_info.txt.gz
genes = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz', sep='\t')
genes.head()

--2026-09-29 20:24:57--  https://ftp.ncbi.nlm.nih.gov/geo/series/GSE92nnn/GSE92742/suppl/GSE92742_Broad_LINCS_gene_info.txt.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.31, 130.14.250.7, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.31|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 216692 (212K) [application/x-gzip]
Saving to: ‘/content/drive/MyDrive/longevity_project/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz’

GSE92742_Broad_LINC 100%[===================>] 211.61K  --.-KB/s    in 0.04s   

2026-09-29 20:24:57 (5.32 MB/s) - ‘/content/drive/MyDrive/longevity_project/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz’ saved [216692/216692]



,pr_gene_id,pr_gene_symbol,pr_gene_title,pr_is_lm,pr_is_bing
0,780,DDR1,discoidin domain receptor tyrosine kinase 1,1,1
1,7849,PAX8,paired box 8,1,1
2,2978,GUCA1A,guanylate cyclase activator 1A,0,0
3,2049,EPHB3,EPH receptor B3,0,1
4,2101,ESRRA,estrogen related receptor alpha,0,1


In [13]:
check = pd.read_parquet(f'{BASE}/data/processed/vcap_signatures.parquet')
print(check.shape)
check.iloc[:3, :5]

(34753, 12328)


,5720,466,6009,2309,387
CPC002_VCAP_6H:BRD-K26548821-001-01-8:10,-0.619494,0.561172,0.315348,-0.303915,0.574344
CPC001_VCAP_6H:BRD-A95445494-001-02-9:10,0.069401,0.693944,-0.472798,-0.525526,-0.062443
CPC001_VCAP_6H:BRD-K05906022-001-02-8:10,0.145600,-0.975817,0.189432,-0.652556,-0.474513
